# DPO vs SFT for Medical Reasoning — End-to-End Runner

**Pre-registration lock:** all decisions in this notebook derive from `PLAN.md` and `configs/`. The statistical protocol in `src/stats.py` was committed BEFORE any adapted model was trained. Any post-hoc change is flagged in `REPORT.md` as a deviation.

**How this notebook is organized:**

The notebook has 9 sections, in the same order as `PLAN.md` §7 (the week-by-week timeline). Each section is independently resumable — you can close Colab, come back next day, and pick up where you left off. Model checkpoints and cached outputs live in Google Drive under `MyDrive/dpo-sft-medical/`, so a runtime disconnect never loses more than the currently-executing stage.

Sections:
1. Environment and Drive mount (~5 min)
2. Repo checkout and dependency install (~3 min)
3. Dataset verification checklist (Week 1 hard gate)
4. Baseline evaluations on the base model
5. Data construction (SFT dataset + DPO preference pairs)
6. Training — 3 SFT seeds + 3 DPO seeds
7. Evaluation — all 6 adapted models on medical + safety benchmarks
8. Mechanistic analyses — KL divergence + LoRA Frobenius audit
9. Pre-registered statistical analysis (`src/stats.py`) + figures

**Estimated total wall-clock:** ~41 hours on A100 40GB (see `PLAN.md` §6.1). Runs across ~5 sessions with a fully resumable pipeline.


## 1. Environment and Google Drive

Mounts Drive so all checkpoints and cached outputs persist across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/dpo-sft-medical'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Drive workspace: {DRIVE_ROOT}")

# GPU sanity check
!nvidia-smi | head -20


## 2. Repository and dependencies

Clone your repo (replace `USERNAME` with your GitHub handle) and install requirements.

**Important:** the repo path is a symlink into Drive so any commit you make from the notebook survives disconnect.

In [ ]:
REPO_URL = "https://github.com/USERNAME/dpo-vs-sft-medical-safety.git"  # <-- EDIT
REPO_LOCAL = "/content/repo"

import os, subprocess
if not os.path.isdir(REPO_LOCAL):
    subprocess.run(["git", "clone", REPO_URL, REPO_LOCAL], check=True)
os.chdir(REPO_LOCAL)
print("cwd:", os.getcwd())
!git log --oneline -5


In [ ]:
!pip install -q -r requirements.txt

# Verify src/ is importable
import sys
sys.path.insert(0, REPO_LOCAL)
from src import stats
print("stats.py loaded; pre-registered constants:")
print(f"  SEEDS = {stats.SEEDS}")
print(f"  BOOTSTRAP_N = {stats.BOOTSTRAP_N}")
print(f"  ALPHA_BONFERRONI = {stats.ALPHA_BONFERRONI}")
print(f"  H1_MIN_DELTA_PP = {stats.H1_MIN_DELTA_PP}")


In [ ]:
# Run the pre-registered stats test suite — MUST pass before we start training
!cd $REPO_LOCAL && python -m pytest tests/test_stats.py -v --tb=short 2>&1 | tail -30


## 3. Dataset verification (Week 1 hard gate)

**This section is a hard gate.** If any of the checks below fail, STOP and resolve the access issue before proceeding to Section 4. Per `PLAN.md` §7, discovering a dataset issue in Week 3 is the study's largest schedule risk.

Checklist:
- [ ] Llama-3.2-3B-Instruct downloadable and loadable in 4-bit
- [ ] Qwen2.5-3B-Instruct verified as usable fallback
- [ ] MedMCQA train and validation splits loadable
- [ ] MedQA-USMLE test split accessible
- [ ] AdvBench-Harmful accessible; else HarmBench-standard verified as fallback
- [ ] MedSafetyBench accessible; else hand-curated 200-prompt set drafted
- [ ] Llama-Guard-3-8B downloadable

In [ ]:
# HuggingFace login — gated models (Llama, Llama-Guard) require it
from huggingface_hub import login
login()  # paste your HF token when prompted


In [ ]:
# ---- 3a. Base model + fallback ----
from transformers import AutoTokenizer

for name in ["meta-llama/Llama-3.2-3B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]:
    try:
        tok = AutoTokenizer.from_pretrained(name)
        print(f"OK  {name}  (vocab={tok.vocab_size})")
    except Exception as e:
        print(f"FAIL {name}: {e}")


In [ ]:
# ---- 3b. Medical datasets ----
from datasets import load_dataset

for spec in [
    ("openlifescienceai/medmcqa", None, "train"),
    ("openlifescienceai/medmcqa", None, "validation"),
    ("bigbio/med_qa", "med_qa_en_bigbio_qa", "test"),
]:
    name, config, split = spec
    try:
        ds = load_dataset(name, config, split=split, streaming=True)
        first = next(iter(ds))
        print(f"OK  {name} [{config}] [{split}] — fields: {list(first.keys())[:6]}")
    except Exception as e:
        print(f"FAIL {name} [{config}] [{split}]: {str(e)[:120]}")


In [ ]:
# ---- 3c. Safety benchmarks (probe availability; do NOT fully download yet) ----
# AdvBench / HarmBench / MedSafetyBench access can drift; try each and record.

candidates = [
    ("walledai/AdvBench", None, "train"),
    ("walledai/HarmBench", "standard", "train"),
    ("MedSafetyBench/MedSafetyBench", None, "test"),  # placeholder — verify actual repo path
]
for name, config, split in candidates:
    try:
        ds = load_dataset(name, config, split=split, streaming=True)
        first = next(iter(ds))
        print(f"OK  {name} — sample fields: {list(first.keys())[:6]}")
    except Exception as e:
        print(f"FAIL {name}: {str(e)[:150]}")

# If MedSafetyBench fails, use configs/discordant_rubric.md as the model
# and construct the hand-curated 200-prompt set from published taxonomies.
# The set must be committed to data/safety_medical.jsonl BEFORE Week 2.


In [ ]:
# ---- 3d. Judge model ----
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

try:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16)
    tok = AutoTokenizer.from_pretrained("meta-llama/Llama-Guard-3-8B")
    # Just probe the tokenizer; loading the full model here wastes disk.
    print(f"OK  Llama-Guard-3-8B tokenizer loaded (vocab={tok.vocab_size})")
except Exception as e:
    print(f"FAIL Llama-Guard-3-8B: {e}")


## 4. Baseline evaluations on the base model

Cached to Drive: these run ONCE and are reused for every subsequent comparison.

- MedMCQA validation accuracy (base model)
- MedQA test accuracy (base model)
- Safety refusal rate on general + medical safety sets (base model)

In [ ]:
# TODO(WEEK 1): implement using the same generate+judge pipeline as src/eval_medical.py
# and src/eval_safety.py, but with the raw base model (no adapter). Write outputs to:
#   {DRIVE_ROOT}/baselines/medmcqa_base.jsonl
#   {DRIVE_ROOT}/baselines/medqa_base.jsonl
#   {DRIVE_ROOT}/baselines/safety_general_base.jsonl
#   {DRIVE_ROOT}/baselines/safety_medical_base.jsonl
#
# Ceiling-effect check per PLAN.md risk table:
# if base_safety_rate > 0.95 on both benchmarks, PIVOT to a harder safety subset
# and disclose the pivot in REPORT.md.
print("Baseline eval — implement in Week 1 per PLAN.md §7 M3.")


## 5. Data construction (Week 2)

**Locked artifacts written by this section (do not overwrite after commit):**
- `data/adaptation_ids.json` — frozen 4,000-question sample
- `data/sft_dataset.jsonl` — SFT training data
- `data/dpo_pairs.jsonl` — DPO preference pairs
- `data/safety_general.jsonl` — 200 general safety prompts
- `data/safety_medical.jsonl` — 200 medical safety prompts
- `data/kl_holdout.jsonl` — 200 held-out prompts for KL evaluation

In [ ]:
import json, yaml
from pathlib import Path
from datasets import load_dataset
from src.data_prep import freeze_adaptation_ids, build_sft_dataset

seeds = yaml.safe_load(open('configs/seeds.yaml'))
sft_cfg = yaml.safe_load(open('configs/sft.yaml'))

# ---- Freeze adaptation IDs (idempotent) ----
train = load_dataset('openlifescienceai/medmcqa', split='train')
train_by_id = {r['id']: r for r in train}
adaptation_ids = freeze_adaptation_ids(
    medmcqa_train_ids=list(train_by_id.keys()),
    out_path=Path('data/adaptation_ids.json'),
    seed=seeds['adaptation_subset_seed'],
)
print(f"Adaptation set: {len(adaptation_ids)} IDs, first 3 = {adaptation_ids[:3]}")

# ---- Build SFT dataset ----
build_sft_dataset(
    medmcqa_train_by_id=train_by_id,
    adaptation_ids=adaptation_ids,
    out_path=Path('data/sft_dataset.jsonl'),
    instruction=sft_cfg['data']['response_prefix_instruction'],
)
print("SFT dataset built.")


In [ ]:
# ---- Build DPO preference pairs ----
# Requires the base model loaded for generation. This step is the slowest of
# Section 5 (~3 h on A100 40GB).

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from src.data_prep import build_dpo_pairs

BASE = sft_cfg['base_model']['name']
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16,
)
model.eval()

def base_generate(prompt: str, seed: int) -> str:
    set_seed(seed)
    messages = [{"role": "user", "content": prompt}]
    input_ids = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids, max_new_tokens=256, do_sample=True,
            temperature=0.7, top_p=0.95, pad_token_id=tok.pad_token_id,
        )
    return tok.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()

build_dpo_pairs(
    medmcqa_train_by_id=train_by_id,
    adaptation_ids=adaptation_ids,
    out_path=Path('data/dpo_pairs.jsonl'),
    instruction=sft_cfg['data']['response_prefix_instruction'],
    base_model_generate_fn=base_generate,
    seed=seeds['adaptation_subset_seed'],
)
print("DPO pairs built.")

# Manual spot check (M5 in PLAN §7)
with open('data/dpo_pairs.jsonl') as f:
    for i, line in enumerate(f):
        if i >= 3: break
        r = json.loads(line)
        print(f"\n--- Pair {i} ---")
        print(f"PROMPT: {r['prompt'][:200]}...")
        print(f"CHOSEN: {r['chosen'][:200]}")
        print(f"REJECTED: {r['rejected'][:200]}")


In [ ]:
# ---- Safety prompt sets and KL holdout ----
# TODO(WEEK 2 M4/M5): implement per PLAN §4.4 and §5.3.
# - data/safety_general.jsonl: 200 prompts sampled from AdvBench (seed = safety_prompt_seed)
# - data/safety_medical.jsonl: 200 prompts from MedSafetyBench OR hand-curated
# - data/kl_holdout.jsonl: 100 MedMCQA-style + 100 Alpaca-style prompts
print("Safety and KL sets — implement Week 2, commit before any training.")


## 6. Training — 3 SFT seeds + 3 DPO seeds (Week 3)

Fully resumable. Each run writes a `TRAINING_COMPLETE` marker; re-running skips completed models.

In [ ]:
from pathlib import Path
from src.train_sft import train_one_seed as train_sft
from src.train_dpo import train_one_seed as train_dpo

sft_cfg = yaml.safe_load(open('configs/sft.yaml'))
dpo_cfg = yaml.safe_load(open('configs/dpo.yaml'))
SEEDS = yaml.safe_load(open('configs/seeds.yaml'))['training_seeds']

CKPT_ROOT = Path(DRIVE_ROOT) / "checkpoints"

for seed in SEEDS:
    print(f"\n===== SFT seed {seed} =====")
    train_sft(
        sft_config=sft_cfg,
        seed=seed,
        dataset_path=Path('data/sft_dataset.jsonl'),
        output_dir=CKPT_ROOT / f"sft_seed{seed}",
    )

for seed in SEEDS:
    print(f"\n===== DPO seed {seed} =====")
    train_dpo(
        dpo_config=dpo_cfg,
        seed=seed,
        dataset_path=Path('data/dpo_pairs.jsonl'),
        output_dir=CKPT_ROOT / f"dpo_seed{seed}",
    )


## 7. Evaluation — all 6 adapted models (Week 4)

Writes per-model JSONL files into `results/`. The stats module reads them by filename convention.

In [ ]:
from src.eval_medical import evaluate_one_model
from src.eval_safety import generate_responses, judge_responses

RESULTS = Path('results')
RESULTS.mkdir(exist_ok=True)

# Prepare benchmark records once
# TODO(WEEK 4): load MedMCQA validation and MedQA test, format to
#   {prompt_id, prompt, gold_letter} JSONL. Cache under data/.

medmcqa_val = json.load(open('data/medmcqa_val.json'))   # TODO: build in Week 1
medqa_test  = json.load(open('data/medqa_test.json'))    # TODO: build in Week 1
safety_gen  = [json.loads(l) for l in open('data/safety_general.jsonl')]
safety_med  = [json.loads(l) for l in open('data/safety_medical.jsonl')]

INSTR = sft_cfg['data']['response_prefix_instruction']

for condition in ["sft", "dpo"]:
    for seed in SEEDS:
        model_dir = CKPT_ROOT / f"{condition}_seed{seed}"
        print(f"\n----- eval {condition} seed {seed} -----")

        # Medical
        evaluate_one_model(model_dir, medmcqa_val,
                           RESULTS / f"medmcqa_{condition}_seed{seed}.jsonl",
                           instruction=INSTR)
        evaluate_one_model(model_dir, medqa_test,
                           RESULTS / f"medqa_{condition}_seed{seed}.jsonl",
                           instruction=INSTR)

        # Safety — generate then judge
        gen_seed = yaml.safe_load(open('configs/seeds.yaml'))['generation_seed']
        for domain, prompts in [("general", safety_gen), ("medical", safety_med)]:
            gen_path = RESULTS / f"safety_{domain}_{condition}_seed{seed}.gen.jsonl"
            out_path = RESULTS / f"safety_{domain}_{condition}_seed{seed}.jsonl"
            generate_responses(model_dir, prompts, gen_path, generation_seed=gen_seed)
            judge_responses(gen_path, out_path)


In [ ]:
# ---- Judge-human κ spot-check (PLAN §4.5) ----
# ONE-TIME check, before Section 9 runs the pooled analysis.
# Randomly sample 100 judged responses from seed 42, both conditions.
# Manually annotate safe/unsafe, compute Cohen's κ, record in results/judge_kappa.json.
# If κ < 0.7, switch to full manual annotation of all 400 safety responses.
print("Judge-κ spot-check — do BEFORE running Section 9. See PLAN §4.5.")


## 8. Mechanistic analyses (Week 5)

Cheap (~2 h total). Runs after all 6 models are trained and evaluated.

In [ ]:
from src.mech_kl import compute_kl
from src.mech_norms import audit_lora_norms

BASE = sft_cfg['base_model']['name']
holdout = Path('data/kl_holdout.jsonl')

for condition in ["sft", "dpo"]:
    for seed in SEEDS:
        md = CKPT_ROOT / f"{condition}_seed{seed}"
        compute_kl(md, BASE, holdout, RESULTS / f"kl_{condition}_seed{seed}.json")
        audit_lora_norms(md, BASE, RESULTS / f"lora_norms_{condition}_seed{seed}.json")


## 9. Pre-registered analysis and figures (Week 5–6)

Runs `src/stats.py` end-to-end and produces `figures/stats_report.json` plus the plots that go into REPORT.md.

In [ ]:
# The moment of pre-registration payoff.
!cd $REPO_LOCAL && python -m src.stats --results-dir results --output figures/stats_report.json


In [ ]:
# Load and pretty-print the top-line decisions
import json
report = json.load(open('figures/stats_report.json'))

print("=" * 72)
print("PRIMARY DECISIONS")
print("=" * 72)
for name in ["utility_parity", "H1_general", "H1_medical", "H2"]:
    d = report['decisions'][name]
    status = "SUPPORTED" if d['supported'] else "NOT SUPPORTED"
    print(f"\n[{name}] {status}")
    for r in d['reasons']:
        print(f"  · {r}")


In [ ]:
# ---- Figures ----
# 1. Utility–safety Pareto plot across the 6 runs
# 2. Per-seed safety Δ bar chart with 95% CIs
# 3. Per-module LoRA Frobenius norm comparison (SFT vs DPO)
# 4. KL divergence density plot
# 5. Discordant-example category breakdown (after Section 5.5 manual coding)
#
# TODO(WEEK 5): matplotlib code here — every figure saved to figures/ as PNG + PDF.
print("Figures — implement Week 5, styled to match REPORT.md.")


## Done.

Next: draft `REPORT.md` per the template committed alongside this notebook. The report cites `figures/stats_report.json` directly and states each decision against the pre-registered rule.